In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_49.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_67_1.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_90.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_77.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_20_1.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_66.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_54.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_106_1.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_42.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_81.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_72.wav
/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test/audio_107.wav
/kaggle/input/shl-in

In [2]:
# ===============================
# Environment Setup (RUN FIRST)
# ===============================
!pip install -U openai-whisper
!pip install language-tool-python==2.7.1
!pip install spacy happytransformer lightgbm librosa soundfile
!python -m spacy download en_core_web_sm
!pip install happytransformer
!pip install -q openai-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=945ed1c2fe5e2538f4471a0ba59dfd49a828792d1ef85dbcf4c593cd9e7d0e3d
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 7.7 MB/s eta 0:00:00
  At

In [3]:
import os
import librosa
import soundfile as sf
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

import whisper
import language_tool_python
import spacy
from happytransformer import HappyTextToText, TTSettings

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

2025-12-20 13:32:27.187537: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766237547.439349      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766237547.513456      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766237548.141473      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766237548.141519      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766237548.141522      17 computation_placer.cc:177] computation placer alr

In [4]:
import pandas as pd

CSV_PATH = '/kaggle/input/shl-intern-hiring-assessment-2025/dataset/csvs/train.csv'
AUDIO_DIR = '/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/train'

train_df = pd.read_csv(CSV_PATH)
train_df.columns = ['filename', 'label']

print("Train samples:", len(train_df))
train_df.head()

Train samples: 409


,filename,label
0,audio_173,3.0
1,audio_138,3.0
2,audio_127,2.0
3,audio_95,2.0
4,audio_73,3.5


In [5]:
import os

PROCESSED_DIR = "/kaggle/working/processed_audio"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("PROCESSED_DIR created:", os.path.exists(PROCESSED_DIR))

PROCESSED_DIR created: True


In [6]:
import os
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

# Define directories INSIDE the cell
AUDIO_DIR = '/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/train'
PROCESSED_DIR = '/kaggle/working/processed_audio'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Define preprocessing function INSIDE the cell
def preprocess_audio(file_path, save_path, target_sr=16000):
    y, sr = librosa.load(file_path, sr=None)
    if sr != target_sr:
        y = librosa.resample(y=y, orig_sr=sr, target_sr=target_sr)
    y = y / max(abs(y)) if max(abs(y)) > 0 else y
    sf.write(save_path, y, target_sr)

# Preprocess all training audio
processed_count = 0

for fname in tqdm(train_df['filename']):
    in_path = os.path.join(AUDIO_DIR, fname)
    out_path = os.path.join(PROCESSED_DIR, fname)

    if os.path.exists(in_path):
        preprocess_audio(in_path, out_path)
        processed_count += 1

print("Processed files:", processed_count)
print("Sample processed files:", os.listdir(PROCESSED_DIR)[:5])


  0%|          | 0/409 [00:00<?, ?it/s]

Processed files: 0
Sample processed files: []


In [7]:
# Ensure .wav extension exists
train_df['filename'] = train_df['filename'].apply(
    lambda x: x if x.endswith('.wav') else f"{x}.wav"
)

# Verify
print(train_df['filename'].head())


0    audio_173.wav
1    audio_138.wav
2    audio_127.wav
3     audio_95.wav
4     audio_73.wav
Name: filename, dtype: object


In [8]:
import os

missing_files = [
    f for f in train_df['filename']
    if not os.path.exists(os.path.join('/kaggle/working/processed_audio', f))
]

print("Missing processed files:", len(missing_files))
print("Examples:", missing_files[:5])


Missing processed files: 409
Examples: ['audio_173.wav', 'audio_138.wav', 'audio_127.wav', 'audio_95.wav', 'audio_73.wav']


In [9]:
# Check what filenames look like in CSV
print(train_df['filename'].head())

# Check what files actually exist
print("\nSample files in processed_audio:")
print(os.listdir(PROCESSED_DIR)[:10])


0    audio_173.wav
1    audio_138.wav
2    audio_127.wav
3     audio_95.wav
4     audio_73.wav
Name: filename, dtype: object

Sample files in processed_audio:
[]


In [10]:
# Add .wav extension if missing
train_df['filename'] = train_df['filename'].apply(
    lambda x: x if x.endswith('.wav') else f"{x}.wav"
)

# Verify
print(train_df['filename'].head())


0    audio_173.wav
1    audio_138.wav
2    audio_127.wav
3     audio_95.wav
4     audio_73.wav
Name: filename, dtype: object


In [11]:
# Check if all processed audio files exist
missing_files = []

for fname in train_df['filename']:
    if not os.path.exists(os.path.join(PROCESSED_DIR, fname)):
        missing_files.append(fname)

print("Total training files:", len(train_df))
print("Missing processed audio files:", len(missing_files))
print("Example missing files:", missing_files[:5])


Total training files: 409
Missing processed audio files: 409
Example missing files: ['audio_173.wav', 'audio_138.wav', 'audio_127.wav', 'audio_95.wav', 'audio_73.wav']


In [12]:
# Clear old processed_audio directory
import shutil

shutil.rmtree(PROCESSED_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Old processed_audio cleared.")


Old processed_audio cleared.


In [13]:
def preprocess_audio(file_path, save_path, sr=16000):
    y, orig_sr = librosa.load(file_path, sr=None)
    if orig_sr != sr:
        y = librosa.resample(y, orig_sr, sr)
    y = y / max(abs(y))
    sf.write(save_path, y, sr)

print("preprocess_audio function defined.")


preprocess_audio function defined.


In [14]:
def preprocess_audio(file_path, save_path, sr=16000):
    y, orig_sr = librosa.load(file_path, sr=None)

    if orig_sr != sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)

    y = y / max(abs(y))
    sf.write(save_path, y, sr)

print("preprocess_audio function redefined (librosa-compatible).")


preprocess_audio function redefined (librosa-compatible).


In [15]:
for filename in tqdm(train_df['filename']):
    in_path = os.path.join(AUDIO_DIR, filename)
    out_path = os.path.join(PROCESSED_DIR, filename)
    preprocess_audio(in_path, out_path)

print("Audio preprocessing completed successfully.")


  0%|          | 0/409 [00:00<?, ?it/s]

/tmp/ipykernel_17/2534552399.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, orig_sr = librosa.load(file_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_17/2534552399.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, orig_sr = librosa.load(file_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_17/2534552399.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, orig_sr = librosa.load(file_path, sr=None)
/usr/local/lib/python3.12/dist-packa

Audio preprocessing completed successfully.


In [16]:
# Verify processed audio files again
missing_files = []

for fname in train_df['filename']:
    if not os.path.exists(os.path.join(PROCESSED_DIR, fname)):
        missing_files.append(fname)

print("Total training files:", len(train_df))
print("Missing processed audio files:", len(missing_files))
print("Example missing files:", missing_files[:5])


Total training files: 409
Missing processed audio files: 0
Example missing files: []


In [17]:
# Load Whisper model
model_whisper = whisper.load_model("base")

# Transcribe processed audio
transcripts = []

for fname in tqdm(train_df['filename']):
    audio_path = os.path.join(PROCESSED_DIR, fname)
    result = model_whisper.transcribe(
        audio_path,
        language="en",
        fp16=False   # IMPORTANT for Kaggle stability
    )
    transcripts.append(result["text"])

# Attach transcripts
train_df['transcript'] = transcripts

print("Whisper transcription completed.")


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 166MiB/s]


  0%|          | 0/409 [00:00<?, ?it/s]

Whisper transcription completed.


In [18]:
# Save transcripts immediately (important)
train_df.to_csv('/kaggle/working/train_with_transcripts.csv', index=False)

print("Transcripts saved safely.")
train_df[['filename', 'transcript']].head()


Transcripts saved safely.


,filename,transcript
0,audio_173.wav,"My favorite place to visit will be Japan, bec..."
1,audio_138.wav,I love to reading on my hobby such reading. E...
2,audio_127.wav,My favorite place to visit is the Mullahites ...
3,audio_95.wav,I am going to tell about my hobby. And my hob...
4,audio_73.wav,This is a tough one. So my bestie of my life ...


In [19]:
import re

FILLERS = ['uh', 'um', 'erm', 'you know', 'like', 'i mean', 'hmm', 'ah', 'uhh', 'huh']

def clean_transcript(text):
    text = text.lower()
    text = re.sub(r'\b(?:' + '|'.join(FILLERS) + r')\b', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([?.!,"])', r'\1', text)
    return text.strip()

train_df['cleaned_transcript'] = train_df['transcript'].astype(str).apply(clean_transcript)

print("Transcript cleaning completed.")
train_df[['transcript', 'cleaned_transcript']].head()


Transcript cleaning completed.


,transcript,cleaned_transcript
0,"My favorite place to visit will be Japan, bec...","my favorite place to visit will be japan, beca..."
1,I love to reading on my hobby such reading. E...,i love to reading on my hobby such reading. em...
2,My favorite place to visit is the Mullahites ...,my favorite place to visit is the mullahites n...
3,I am going to tell about my hobby. And my hob...,i am going to tell about my hobby. and my hobb...
4,This is a tough one. So my bestie of my life ...,this is a tough one. so my bestie of my life i...


In [20]:
import language_tool_python

# Use public LanguageTool server (NO local download, NO Java)
tool = language_tool_python.LanguageToolPublicAPI('en-US')

print("LanguageTool initialized using remote API.")


LanguageTool initialized using remote API.


In [21]:
import spacy
from happytransformer import HappyTextToText, TTSettings
from tqdm.notebook import tqdm

# NLP + GEC models
nlp = spacy.load("en_core_web_sm")

happy_tt = HappyTextToText("T5", "vennify/t5-base-grammar-correction")
args = TTSettings(num_beams=5, min_length=1)

# Feature containers
avg_sent_lengths = []
pos_diversities = []
word_counts = []
gec_edits = []
gec_rates = []

for text in tqdm(train_df['cleaned_transcript']):
    # NLP features
    doc = nlp(text)
    sent_lens = [len(sent) for sent in doc.sents]
    pos_tags = [token.pos_ for token in doc if token.pos_ != 'SPACE']

    avg_sent_lengths.append(sum(sent_lens) / len(sent_lens) if sent_lens else 0)
    pos_diversities.append(len(set(pos_tags)))

    # Word count
    words = text.split()
    wc = len(words)
    word_counts.append(wc)

    # Grammar correction edits
    corrected = happy_tt.generate_text("grammar: " + text, args=args).text
    corr_words = corrected.split()

    edits = sum(1 for o, c in zip(words, corr_words) if o != c)
    edits += abs(len(words) - len(corr_words))

    gec_edits.append(edits)
    gec_rates.append(edits / max(1, wc))

# Attach features (NO LanguageTool)
train_df['avg_sentence_length'] = avg_sent_lengths
train_df['pos_diversity'] = pos_diversities
train_df['word_count'] = word_counts
train_df['gec_edits'] = gec_edits
train_df['gec_edit_rate'] = gec_rates

print("Feature extraction completed (LanguageTool removed).")
train_df.head()


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

  0%|          | 0/409 [00:00<?, ?it/s]

num_beams is deprecated and will have no effect in version 4.0.0
Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
num_beams is deprecated and will have no effect in version 4.0.0
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
num_beams is deprecated and will have no effect in version 4.0.0
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
num_beams is deprecated and will hav

Feature extraction completed (LanguageTool removed).


,filename,label,transcript,cleaned_transcript,avg_sentence_length,pos_diversity,word_count,gec_edits,gec_edit_rate
0,audio_173.wav,3.0,"My favorite place to visit will be Japan, bec...","my favorite place to visit will be japan, beca...",15.000000,14,89,54,0.606742
1,audio_138.wav,3.0,I love to reading on my hobby such reading. E...,i love to reading on my hobby such reading. em...,26.833333,15,153,153,1.000000
2,audio_127.wav,2.0,My favorite place to visit is the Mullahites ...,my favorite place to visit is the mullahites n...,10.000000,12,61,2,0.032787
3,audio_95.wav,2.0,I am going to tell about my hobby. And my hob...,i am going to tell about my hobby. and my hobb...,16.166667,12,86,29,0.337209
4,audio_73.wav,3.5,This is a tough one. So my bestie of my life ...,this is a tough one. so my bestie of my life i...,13.700000,13,120,53,0.441667


In [22]:
import os
from tqdm.notebook import tqdm
import whisper

# Define paths INSIDE the cell
PROCESSED_DIR = '/kaggle/working/processed_audio'

# Load Whisper model
model_whisper = whisper.load_model("base")

transcripts = []

for fname in tqdm(train_df['filename']):
    audio_path = os.path.join(PROCESSED_DIR, fname)

    result = model_whisper.transcribe(
        audio_path,
        language='en',
        fp16=False
    )

    transcripts.append(result['text'])

# Add transcript column
train_df['transcript'] = transcripts

print("✅ Whisper transcription completed.")
train_df[['filename', 'transcript']].head()


  0%|          | 0/409 [00:00<?, ?it/s]

✅ Whisper transcription completed.


,filename,transcript
0,audio_173.wav,"My favorite place to visit will be Japan, bec..."
1,audio_138.wav,I love to reading on my hobby such reading. E...
2,audio_127.wav,My favorite place to visit is the Mullahites ...
3,audio_95.wav,I am going to tell about my hobby. And my hob...
4,audio_73.wav,This is a tough one. So my bestie of my life ...


In [23]:
import re

FILLERS = ['uh', 'um', 'erm', 'you know', 'like', 'i mean', 'hmm', 'ah', 'uhh', 'huh']

def clean_transcript(text):
    text = text.lower()
    text = re.sub(r'\b(?:' + '|'.join(FILLERS) + r')\b', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([?.!,"])', r'\1', text)
    return text.strip()

train_df['cleaned_transcript'] = train_df['transcript'].astype(str).apply(clean_transcript)

print("✅ Transcript cleaning completed.")
train_df[['transcript', 'cleaned_transcript']].head()


✅ Transcript cleaning completed.


,transcript,cleaned_transcript
0,"My favorite place to visit will be Japan, bec...","my favorite place to visit will be japan, beca..."
1,I love to reading on my hobby such reading. E...,i love to reading on my hobby such reading. em...
2,My favorite place to visit is the Mullahites ...,my favorite place to visit is the mullahites n...
3,I am going to tell about my hobby. And my hob...,i am going to tell about my hobby. and my hobb...
4,This is a tough one. So my bestie of my life ...,this is a tough one. so my bestie of my life i...


In [24]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

avg_sentence_length = []
pos_diversity = []
word_count = []

for text in train_df['cleaned_transcript']:
    doc = nlp(text)

    # Average sentence length
    sent_lengths = [len(sent) for sent in doc.sents]
    avg_sentence_length.append(
        sum(sent_lengths) / len(sent_lengths) if sent_lengths else 0
    )

    # POS diversity
    pos_tags = [token.pos_ for token in doc if token.pos_ != 'SPACE']
    pos_diversity.append(len(set(pos_tags)))

    # Word count
    word_count.append(len(text.split()))

# Add features to dataframe
train_df['avg_sentence_length'] = avg_sentence_length
train_df['pos_diversity'] = pos_diversity
train_df['word_count'] = word_count

print("✅ Basic text features extracted.")
train_df[['avg_sentence_length', 'pos_diversity', 'word_count']].head()


✅ Basic text features extracted.


,avg_sentence_length,pos_diversity,word_count
0,15.000000,14,89
1,26.833333,15,153
2,10.000000,12,61
3,16.166667,12,86
4,13.700000,13,120


In [25]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# Feature matrix and target
features = ['avg_sentence_length', 'pos_diversity', 'word_count']
X = train_df[features]
y = train_df['label']

# LightGBM Regressor
lgb_model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

# Train on full training data
lgb_model.fit(X, y)

# Training predictions
train_preds = lgb_model.predict(X)

# Metrics (MANDATORY)
rmse = np.sqrt(mean_squared_error(y, train_preds))
pearson = pearsonr(y, train_preds)[0]

print(f"✅ Training RMSE: {rmse:.3f}")
print(f"✅ Training Pearson Correlation: {pearson:.3f}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001088 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 221
[LightGBM] [Info] Number of data points in the train set: 409, number of used features: 3
[LightGBM] [Info] Start training from score 2.910758
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

In [26]:
import os
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load test CSV
# -----------------------------
TEST_CSV_PATH = '/kaggle/input/shl-intern-hiring-assessment-2025/dataset/csvs/test.csv'
TEST_AUDIO_DIR = '/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test'
PROCESSED_TEST_DIR = '/kaggle/working/processed_test_audio'

os.makedirs(PROCESSED_TEST_DIR, exist_ok=True)

test_df = pd.read_csv(TEST_CSV_PATH)
test_df.columns = ['filename']

# Ensure .wav extension
test_df['filename'] = test_df['filename'].apply(
    lambda x: x if x.endswith('.wav') else f"{x}.wav"
)

print("Test samples:", len(test_df))


# -----------------------------
# 2. Preprocess test audio
# -----------------------------
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

def preprocess_audio(file_path, save_path, target_sr=16000):
    y, sr = librosa.load(file_path, sr=None)
    if sr != target_sr:
        y = librosa.resample(y=y, orig_sr=sr, target_sr=target_sr)
    y = y / max(abs(y)) if max(abs(y)) > 0 else y
    sf.write(save_path, y, target_sr)

for fname in tqdm(test_df['filename']):
    in_path = os.path.join(TEST_AUDIO_DIR, fname)
    out_path = os.path.join(PROCESSED_TEST_DIR, fname)
    preprocess_audio(in_path, out_path)

print("✅ Test audio preprocessing done.")


# -----------------------------
# 3. Whisper transcription
# -----------------------------
import whisper

model_whisper = whisper.load_model("base")
test_transcripts = []

for fname in tqdm(test_df['filename']):
    audio_path = os.path.join(PROCESSED_TEST_DIR, fname)
    result = model_whisper.transcribe(audio_path, language='en', fp16=False)
    test_transcripts.append(result['text'])

test_df['transcript'] = test_transcripts
print("✅ Test transcription done.")


# -----------------------------
# 4. Clean transcripts
# -----------------------------
import re

FILLERS = ['uh', 'um', 'erm', 'you know', 'like', 'i mean', 'hmm', 'ah', 'uhh', 'huh']

def clean_transcript(text):
    text = text.lower()
    text = re.sub(r'\b(?:' + '|'.join(FILLERS) + r')\b', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([?.!,"])', r'\1', text)
    return text.strip()

test_df['cleaned_transcript'] = test_df['transcript'].astype(str).apply(clean_transcript)


# -----------------------------
# 5. Feature extraction (same as train)
# -----------------------------
import spacy
nlp = spacy.load("en_core_web_sm")

avg_sentence_length = []
pos_diversity = []
word_count = []

for text in test_df['cleaned_transcript']:
    doc = nlp(text)
    sent_lengths = [len(sent) for sent in doc.sents]
    avg_sentence_length.append(
        sum(sent_lengths) / len(sent_lengths) if sent_lengths else 0
    )
    pos_tags = [token.pos_ for token in doc if token.pos_ != 'SPACE']
    pos_diversity.append(len(set(pos_tags)))
    word_count.append(len(text.split()))

test_df['avg_sentence_length'] = avg_sentence_length
test_df['pos_diversity'] = pos_diversity
test_df['word_count'] = word_count


# -----------------------------
# 6. Predict using trained LightGBM
# -----------------------------
features = ['avg_sentence_length', 'pos_diversity', 'word_count']
X_test = test_df[features]

test_preds = lgb_model.predict(X_test)

# Clip to valid range
test_preds = np.clip(test_preds, 0, 5)

test_df['label'] = test_preds


# -----------------------------
# 7. Create submission file
# -----------------------------
submission = test_df[['filename', 'label']]
submission.to_csv('/kaggle/working/submission.csv', index=False)

print("✅ submission.csv created successfully")
submission.head()


Test samples: 197


  0%|          | 0/197 [00:00<?, ?it/s]

/tmp/ipykernel_17/20357853.py:33: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_17/20357853.py:33: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_17/20357853.py:33: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/a

✅ Test audio preprocessing done.


  0%|          | 0/197 [00:00<?, ?it/s]

✅ Test transcription done.
✅ submission.csv created successfully


,filename,label
0,audio_141.wav,2.971168
1,audio_114.wav,2.303637
2,audio_17.wav,2.942789
3,audio_76.wav,3.584549
4,audio_156.wav,2.690033
